# Finance and Banking Clustering Analysis Using Spark SQL

A guided, schema-agnostic workflow for a suitable customer or transaction CSV/Parquet dataset. It does not assume particular column names or invent results. Choose a local file you are authorized to use.

## 1. Configure your dataset
Set `DATA_PATH` to your own CSV or Parquet file. For customer segmentation, each row should represent a customer; transaction files produce transaction clusters.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
DATA_PATH = PROJECT_ROOT / 'data' / 'uploads' / 'replace_with_your_file.csv'
print('Project root:', PROJECT_ROOT)
print('Dataset exists:', DATA_PATH.exists())

## 2. Start local Spark and load the file
The session uses local mode and modest shuffle settings. Java 17 or 21 and a supported Python release are required.

In [ ]:
from src.spark_engine import create_spark, read_dataset, inspect_dataset
spark = create_spark('FinanceBankingNotebook')
df = read_dataset(spark, DATA_PATH)
df.printSchema()
display(df.limit(10).toPandas())

## 3. Inspect quality and use Spark SQL
Review actual detected fields before modeling. Duplicate counts and missing values are reported without silently deleting observations.

In [ ]:
quality = inspect_dataset(df)
quality

In [ ]:
df.createOrReplaceTempView('uploaded_dataset')
spark.sql('SELECT COUNT(*) AS records FROM uploaded_dataset').show()
spark.sql('SELECT * FROM uploaded_dataset LIMIT 10').show(truncate=False)

## 4. Select features and fit K-Means
Choose fields that represent similarity. Exclude customer IDs, names, dates, or outcome labels unless justified. Numeric missing values are imputed with the median; categorical columns are indexed and one-hot encoded. Review cluster sizes along with the silhouette score.

In [ ]:
features = quality['numeric']  # Edit this list after reviewing the detected schema.
K = 3  # Change this value after comparing plausible cluster counts.
print('Candidate numeric features:', features)

In [ ]:
from src.spark_engine import cluster_dataset
clustered, silhouette = cluster_dataset(df, features, k=K)
print(f'Silhouette score for K={K}: {silhouette:.3f}')
clustered.groupBy('prediction').count().orderBy('prediction').show()

## 5. Review and export
Silhouette is a diagnostic and does not prove that segments are useful. Inspect the data and cluster sizes before giving them a business interpretation.

In [ ]:
display(clustered.limit(20).toPandas())
output_path = PROJECT_ROOT / 'exports' / 'clustered_records.csv'
clustered.toPandas().to_csv(output_path, index=False)
print('Saved:', output_path)

In [ ]:
spark.stop()